# Feature Extraction in Audio
## This notebook outlines the concepts behind extracting features from audio

### Feature Extraction

Extracting a set of features that are informative with respect to the desired properties of the original audio data

Low-level features to construct a higher-level of understanding

Need to extract audio features capable of discriminating between different audio classes i.e. speakers, emotions, genres

### Features
- Short-term Windowing (Framing)
    - Energy
    - Spectral Centroid
- Mid-term features
- Spectrogram
- Zero Crossings
- Spectral Rolloff
- MFCC

### Import the library

In [1]:
from pyAudioAnalysis import ShortTermFeatures as aF
from pyAudioAnalysis import audioBasicIO as aIO 
import numpy as np 
import plotly.graph_objects as go 
import plotly
import IPython
from matplotlib import pyplot as plt

### Read Audio file

In [ ]:
# Load the provided audio file
audio_path = "/Users/user/Downloads/count.wav (WAV).mp3"

# Read the audio file using pyAudioAnalysis
[fs, x] = aIO.read_audio_file(audio_path)
print(f"Sampling Rate: {fs} Hz")
print(f"Signal shape: {x.shape}")

### Play the audio file

In [ ]:
# Play the audio file
IPython.display.Audio(audio_path)

### Duration of the audio file

In [ ]:
# Compute duration in seconds
duration = len(x) / fs
print(f"Duration: {duration:.2f} seconds")

### Extract short-term features using a 50msec non-overlapping windows

### Extract features
- Use feature_extraction( )
    - **signal**:         the input signal samples
    - **sampling_rate**:  the sampling freq (in Hz)
    - **window**:         the short-term window size (in samples)
    - **step**:           the short-term window step (in samples)
    - **deltas**:         (opt) True/False if delta features are to be computed
- RETURNS
    - **features** (numpy.ndarray):        contains features (n_feats x numOfShortTermWindows)                     
    - **feature_names** (numpy.ndarray):   contains feature names (n_feats x numOfShortTermWindows)

In [ ]:
# Extract short-term features using 50ms non-overlapping windows
window = int(fs * 0.050)   # 50ms window in samples
step   = int(fs * 0.050)   # 50ms step (non-overlapping)

[features, feature_names] = aF.feature_extraction(x, fs, window, step)
print("Feature extraction complete!")

In [ ]:
print(f"Features array shape: {features.shape}  → (num_features x num_frames)")

### How many frames

In [ ]:
num_frames = features.shape[1]
print(f"Number of frames: {num_frames}")

### How many features

In [ ]:
num_features = features.shape[0]
print(f"Number of features: {num_features}")

### Feature Names

In [ ]:
print("Feature Names:")
for i, name in enumerate(feature_names):
    print(f"  [{i:02d}] {name}")

### Plot Short-term energy

### Time

In [ ]:
# Build a time axis (one value per frame)
t = np.arange(0, features.shape[1]) * step / fs
print(f"Time axis: 0 → {t[-1]:.2f} s  ({len(t)} frames)")

In [ ]:
print(f"Time array shape : {t.shape}")
print(f"Total duration   : {t[-1]:.2f} seconds")

# Energy

In [ ]:
# Energy is feature index 1 in pyAudioAnalysis
energy = features[1, :]
print(f"Energy shape : {energy.shape}")
print(f"Min energy   : {energy.min():.4f}")
print(f"Max energy   : {energy.max():.4f}")

In [ ]:
print(f"Energy — first 5 values: {energy[:5]}")

### Plot Time Vs Energy

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, energy, color='steelblue', linewidth=1.2)
plt.xlabel('Time (s)')
plt.ylabel('Energy')
plt.title('Short-term Energy over Time')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# Spectral Centroid
### Spectral centroid -- centre of mass -- weighted mean of the frequencies present in the sound


In [ ]:
# Spectral Centroid is feature index 3 in pyAudioAnalysis
spectral_centroid_idx = list(feature_names).index('spectral_centroid')
spectral_centroid = features[spectral_centroid_idx, :]
print(f"Spectral Centroid shape: {spectral_centroid.shape}")

### Plot Time Vs Spectral Centroid

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, spectral_centroid, color='darkorange', linewidth=1.2)
plt.xlabel('Time (s)')
plt.ylabel('Spectral Centroid')
plt.title('Spectral Centroid over Time (pyAudioAnalysis)')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Use librosa library

In [17]:
import sklearn
import librosa
import librosa.display

### Read the audio file

In [ ]:
# Load audio using librosa
y, sr = librosa.load(audio_path)
print(f"Signal shape   : {y.shape}")
print(f"Sampling rate  : {sr} Hz")

### Compute Spectral Centroids

In [ ]:
# Compute Spectral Centroids using librosa
spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
print(f"Spectral Centroids shape: {spectral_centroids.shape}")

### Computing the time variable for visualization

In [ ]:
# Convert frame indices to time in seconds
frames = range(len(spectral_centroids))
t_librosa = librosa.frames_to_time(frames, sr=sr)
print(f"Time array shape: {t_librosa.shape}")

### Normalising the spectral centroid for visualisation

In [ ]:
def normalize(x, axis=0):
    return sklearn.preprocessing.minmax_scale(x, axis=axis)

### Plotting the Spectral Centroid along the waveform

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y, sr=sr, alpha=0.4, color='steelblue')
plt.plot(t_librosa, normalize(spectral_centroids), color='r', linewidth=1.5, label='Spectral Centroid (normalised)')
plt.xlabel('Time (s)')
plt.title('Spectral Centroid along Waveform (librosa)')
plt.legend()
plt.tight_layout()
plt.show()

# Mid-term features

### Steps
- Import the library
- Extract mid_features

### Import the library

In [ ]:
from pyAudioAnalysis import MidTermFeatures as mF

### Extract features
- Use mid_feature_extraction( )
    - signal
    - sampling_rate
    - mid_window
    - mid_step
    - short_window
    - short_step

In [ ]:
# Extract mid-term features
mid_window = round(1.0 * fs)    # 1-second mid-term window
mid_step   = round(1.0 * fs)    # 1-second mid-term step
short_win  = round(0.050 * fs)  # 50ms short-term window
short_step = round(0.050 * fs)  # 50ms short-term step

[mt_features, st_features, mt_feature_names] = mF.mid_feature_extraction(
    x, fs, mid_window, mid_step, short_win, short_step
)
print("Mid-term feature extraction complete!")

### Duration, Short-term features, Segment features

In [ ]:
print(f"Duration              : {duration:.2f} seconds")
print(f"Short-term features   : {st_features.shape}  → (features x frames)")
print(f"Mid-term features     : {mt_features.shape}  → (features x segments)")

### Mid-term feature names

In [ ]:
print("Mid-term Feature Names:")
for i, name in enumerate(mt_feature_names):
    print(f"  [{i:02d}] {name}")

# Spectrogram

### Steps
- Import the **librosa** library
- Load the audio file
- Compute Frequencies using FT
- Compute Amplitude
- Plot Frequencies Vs Amplitude as Spectrogram

### Import the library

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

### Load the audio file

In [ ]:
y_spec, sr_spec = librosa.load(audio_path)
print(f"Loaded: {y_spec.shape} samples at {sr_spec} Hz")

#### Sampling rate

In [ ]:
print(f"Sampling Rate: {sr_spec} Hz")

### Compute Frequencies

In [ ]:
# Compute Short-Time Fourier Transform (STFT) to get frequencies
D = librosa.stft(y_spec)
frequencies = librosa.fft_frequencies(sr=sr_spec)
print(f"STFT shape      : {D.shape}  → (frequency bins x time frames)")
print(f"Frequency range : 0 – {frequencies[-1]:.0f} Hz")

### Compute Amplitude

In [ ]:
# Convert amplitude to decibels for better visualisation
amplitude_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
print(f"Amplitude (dB) range: {amplitude_db.min():.1f} to {amplitude_db.max():.1f} dB")

### Display Spectrogram

In [ ]:
plt.figure(figsize=(12, 6))
librosa.display.specshow(amplitude_db, sr=sr_spec, x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (STFT)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

# Zero Crossings

### Steps
- Import the library
- Read the Audio file
- Compute Zero Crossings
- Plot zero crossings

### Import the library

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

### Read the audio file

In [ ]:
y_zc, sr_zc = librosa.load(audio_path)
print(f"Loaded: {y_zc.shape} samples at {sr_zc} Hz")

### Plot the signal

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, color='steelblue')
plt.title('Audio Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

### Compute Zero Crossings

In [ ]:
# Compute zero crossings (True where signal crosses zero)
zero_crossings = librosa.zero_crossings(y_zc, pad=False)
print(f"Total Zero Crossings: {sum(zero_crossings)}")

In [ ]:
# Zero-crossing rate per frame
zcr = librosa.feature.zero_crossing_rate(y_zc)[0]
t_zcr = librosa.frames_to_time(range(len(zcr)), sr=sr_zc)
print(f"Zero-crossing rate shape: {zcr.shape}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t_zcr, zcr, color='green', linewidth=1.2)
plt.title('Zero-Crossing Rate over Time')
plt.xlabel('Time (s)')
plt.ylabel('Zero-Crossing Rate')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# Spectral Rolloff

In [ ]:
# Compute Spectral Rolloff — frequency below which 85% of energy is contained
spectral_rolloff = librosa.feature.spectral_rolloff(y=y_zc, sr=sr_zc, roll_percent=0.85)[0]
t_rolloff = librosa.frames_to_time(range(len(spectral_rolloff)), sr=sr_zc)
print(f"Spectral Rolloff shape: {spectral_rolloff.shape}")

### Plot Spectral Rolloff

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, alpha=0.4, color='steelblue')
plt.plot(t_rolloff, normalize(spectral_rolloff), color='r', linewidth=1.5, label='Spectral Rolloff (normalised)')
plt.title('Spectral Rolloff along Waveform')
plt.xlabel('Time (s)')
plt.legend()
plt.tight_layout()
plt.show()

### Spectral Centroids

In [ ]:
# Compare Spectral Centroid vs Spectral Rolloff
sc_zc = librosa.feature.spectral_centroid(y=y_zc, sr=sr_zc)[0]
t_sc  = librosa.frames_to_time(range(len(sc_zc)), sr=sr_zc)

plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, alpha=0.3, color='steelblue')
plt.plot(t_sc,      normalize(sc_zc),           color='b', linewidth=1.5, label='Spectral Centroid')
plt.plot(t_rolloff, normalize(spectral_rolloff), color='r', linewidth=1.5, label='Spectral Rolloff')
plt.title('Spectral Centroid vs Spectral Rolloff')
plt.xlabel('Time (s)')
plt.legend()
plt.tight_layout()
plt.show()

# MFCC

### Compute MFCC

In [ ]:
# Compute MFCC — 13 coefficients (standard for speech/audio tasks)
mfccs = librosa.feature.mfcc(y=y_zc, sr=sr_zc, n_mfcc=13)
print(f"MFCC shape: {mfccs.shape}  → ({mfccs.shape[0]} coefficients x {mfccs.shape[1]} frames)")

In [ ]:
print(f"Number of MFCC coefficients : {mfccs.shape[0]}")
print(f"Number of frames            : {mfccs.shape[1]}")
print(f"Mean of each coefficient    : {np.mean(mfccs, axis=1).round(2)}")

### Plot MFCC

In [ ]:
plt.figure(figsize=(12, 6))
librosa.display.specshow(mfccs, sr=sr_zc, x_axis='time', cmap='coolwarm')
plt.colorbar()
plt.title('MFCC (13 Coefficients)')
plt.xlabel('Time (s)')
plt.ylabel('MFCC Coefficient')
plt.tight_layout()
plt.show()

# Chromagram

### Compute Chromagram

In [ ]:
# Chromagram — shows the energy distribution across 12 pitch classes (C, C#, D, ...)
chromagram = librosa.feature.chroma_stft(y=y_zc, sr=sr_zc)
print(f"Chromagram shape: {chromagram.shape}  → (12 pitch classes x {chromagram.shape[1]} frames)")

In [ ]:
print(f"Pitch classes : {chromagram.shape[0]}  (C, C#, D, D#, E, F, F#, G, G#, A, A#, B)")
print(f"Time frames   : {chromagram.shape[1]}")

### Plot Chromagram

In [ ]:
plt.figure(figsize=(12, 5))
librosa.display.specshow(chromagram, sr=sr_zc, x_axis='time', y_axis='chroma', cmap='coolwarm')
plt.colorbar()
plt.title('Chromagram')
plt.xlabel('Time (s)')
plt.ylabel('Pitch Class')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 50)
print("  Feature Extraction Complete!")
print("=" * 50)
print("\nFeatures extracted:")
print("  ✓ Short-term Energy")
print("  ✓ Spectral Centroid (pyAudioAnalysis + librosa)")
print("  ✓ Mid-term Features")
print("  ✓ Spectrogram (STFT)")
print("  ✓ Zero Crossings")
print("  ✓ Spectral Rolloff")
print("  ✓ MFCC (13 coefficients)")
print("  ✓ Chromagram (12 pitch classes)")